# 05 — Baseline Flight Disruption Models

Temporal evaluation only: train 2020–2023, validation 2024, test 2025. The primary task is four-class prediction of `normal`, `delay`, `severe_delay`, and `cancelled`. We emphasize macro-F1, balanced accuracy, per-class recall, and log loss because raw accuracy can hide poor performance on rare disruptions.


In [14]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, f1_score, log_loss

cwd = Path.cwd().resolve()
if (cwd / 'data').exists(): ROOT = cwd
elif (cwd.parent / 'data').exists(): ROOT = cwd.parent
else: raise FileNotFoundError(f'Cannot locate project root from {cwd}')
FEATURE_FILE = ROOT / 'data/processed/ogg_model_features_v1.csv.gz'
SCHEMA_FILE = ROOT / 'data/processed/ogg_model_features_v1_schema.json'
print('Feature file exists:', FEATURE_FILE.exists())
print('Schema exists:', SCHEMA_FILE.exists())


Feature file exists: True
Schema exists: True


## 1. Load data and define the temporal split


In [15]:
df = pd.read_csv(FEATURE_FILE, low_memory=False)
target_classes = ['normal','delay','severe_delay','cancelled']
model_df = df[df['disruption_class'].isin(target_classes)].copy()

if 'split' in model_df.columns:
    train_df = model_df[model_df['split'].eq('train')].copy()
    val_df = model_df[model_df['split'].eq('validation')].copy()
    test_df = model_df[model_df['split'].eq('test')].copy()
elif 'year' in model_df.columns:
    train_df = model_df[model_df['year'].between(2020, 2023)].copy()
    val_df = model_df[model_df['year'].eq(2024)].copy()
    test_df = model_df[model_df['year'].eq(2025)].copy()
else:
    raise KeyError("Neither 'split' nor 'year' exists. Re-run Notebook 04.")

display(pd.DataFrame({'split':['train','validation','test'],'rows':[len(train_df),len(val_df),len(test_df)]}))
for name, part in [('train',train_df),('validation',val_df),('test',test_df)]:
    print('\n', name)
    display((part['disruption_class'].value_counts(normalize=True)*100).round(3).to_frame('percent'))


,split,rows
0,train,200274
1,validation,52862
2,test,33623



 train


,percent
disruption_class,
normal,81.101
delay,16.702
cancelled,1.633
severe_delay,0.564



 validation


,percent
disruption_class,
normal,86.871
delay,11.787
cancelled,0.965
severe_delay,0.376



 test


,percent
disruption_class,
normal,86.810
delay,11.885
cancelled,0.803
severe_delay,0.503


## 2. Leakage-safe predictors and robust feature typing

Numeric features are identified using pandas numeric dtypes. Everything else—including pandas string/category extension dtypes and airline codes such as `HA`—is treated as categorical. This prevents strings from entering the median imputer.


In [16]:
exclude = {'disruption_class','split','FlightDate','ogg_sched_dt','weather_dt','Cancelled','CancellationCode','Diverted','DepTime','ArrTime','DepDelay','ArrDelay','DepDelayMinutes','ArrDelayMinutes','CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay'}
feature_cols = [c for c in model_df.columns if c not in exclude and not train_df[c].isna().all()]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c]) and not pd.api.types.is_bool_dtype(train_df[c])]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

type_audit = pd.DataFrame({'feature':feature_cols,'dtype':[str(train_df[c].dtype) for c in feature_cols],'role':['numeric' if c in numeric_cols else 'categorical' for c in feature_cols],'n_unique_train':[train_df[c].nunique(dropna=True) for c in feature_cols]})
display(type_audit)
print('Features:',len(feature_cols),'Numeric:',len(numeric_cols),'Categorical:',len(categorical_cols))

bad_numeric = {}
for c in numeric_cols:
    coerced = pd.to_numeric(train_df[c], errors='coerce')
    bad = train_df[c].notna() & coerced.isna()
    if bad.any(): bad_numeric[c] = train_df.loc[bad,c].astype(str).head(5).tolist()
if bad_numeric: raise ValueError(f'Non-numeric values found in numeric features: {bad_numeric}')
print('Feature-type audit passed.')

X_train,y_train = train_df[feature_cols],train_df['disruption_class']
X_val,y_val = val_df[feature_cols],val_df['disruption_class']
X_test,y_test = test_df[feature_cols],test_df['disruption_class']


,feature,dtype,role,n_unique_train
0,Reporting_Airline,str,categorical,6
1,direction,str,categorical,2
2,other_airport,str,categorical,22
3,Distance,float64,numeric,22
4,CRSDepTime,int64,numeric,766
5,CRSArrTime,int64,numeric,1136
6,sched_hour,int64,numeric,22
7,sched_dow,int64,numeric,7
8,sched_month,int64,numeric,12
9,sched_dayofyear,int64,numeric,366


Features: 27 Numeric: 24 Categorical: 3
Feature-type audit passed.


## 3. Shared preprocessing and majority baseline


In [17]:
numeric_pipe = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical_pipe = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num',numeric_pipe,numeric_cols),('cat',categorical_pipe,categorical_cols)])

majority_class = y_train.value_counts().idxmax()
majority_pred = np.repeat(majority_class,len(y_val))
print('Majority class:',majority_class)
print('Validation accuracy:',round(accuracy_score(y_val,majority_pred),4))
print('Validation balanced accuracy:',round(balanced_accuracy_score(y_val,majority_pred),4))
print('Validation macro-F1:',round(f1_score(y_val,majority_pred,average='macro'),4))


Majority class: normal
Validation accuracy: 0.8687
Validation balanced accuracy: 0.25
Validation macro-F1: 0.2324


## 4. Multinomial logistic-regression baseline

`class_weight='balanced'` is intentional because minority disruption classes matter and model selection uses macro-F1. `lbfgs` with L2 regularization (`C=1.0`) is a stable baseline for this feature set. The weighted model's probabilities should be calibrated later before being presented as user-facing risk probabilities.


In [18]:
logreg = Pipeline([('preprocess',preprocess),('model',LogisticRegression(solver='lbfgs',C=1.0,class_weight='balanced',max_iter=1000,tol=1e-4))])
logreg.fit(X_train,y_train)
val_pred_lr = logreg.predict(X_val)
val_prob_lr = logreg.predict_proba(X_val)
print('Iterations:',logreg.named_steps['model'].n_iter_)
print('Validation accuracy:',round(accuracy_score(y_val,val_pred_lr),4))
print('Validation balanced accuracy:',round(balanced_accuracy_score(y_val,val_pred_lr),4))
print('Validation macro-F1:',round(f1_score(y_val,val_pred_lr,average='macro'),4))
print('Validation log loss:',round(log_loss(y_val,val_prob_lr,labels=logreg.classes_),4))
print(classification_report(y_val,val_pred_lr,labels=target_classes,zero_division=0))


Iterations: [106]
Validation accuracy: 0.3371
Validation balanced accuracy: 0.372
Validation macro-F1: 0.1834
Validation log loss: 1.2713
              precision    recall  f1-score   support

      normal       0.93      0.31      0.47     45922
       delay       0.14      0.53      0.23      6231
severe_delay       0.01      0.60      0.02       199
   cancelled       0.02      0.04      0.02       510

    accuracy                           0.34     52862
   macro avg       0.27      0.37      0.18     52862
weighted avg       0.82      0.34      0.43     52862



## 5. Balanced random-forest baseline


In [19]:
rf_preprocess = ColumnTransformer([('num',SimpleImputer(strategy='median'),numeric_cols),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical_cols)])
rf = Pipeline([('preprocess',rf_preprocess),('model',RandomForestClassifier(n_estimators=300,max_depth=18,min_samples_leaf=3,class_weight='balanced_subsample',n_jobs=-1,random_state=42))])
rf.fit(X_train,y_train)
val_pred_rf = rf.predict(X_val)
val_prob_rf = rf.predict_proba(X_val)
print('Validation accuracy:',round(accuracy_score(y_val,val_pred_rf),4))
print('Validation balanced accuracy:',round(balanced_accuracy_score(y_val,val_pred_rf),4))
print('Validation macro-F1:',round(f1_score(y_val,val_pred_rf,average='macro'),4))
print('Validation log loss:',round(log_loss(y_val,val_prob_rf,labels=rf.classes_),4))
print(classification_report(y_val,val_pred_rf,labels=target_classes,zero_division=0))


Validation accuracy: 0.7106
Validation balanced accuracy: 0.2872
Validation macro-F1: 0.2685
Validation log loss: 0.7139
              precision    recall  f1-score   support

      normal       0.89      0.77      0.82     45922
       delay       0.18      0.38      0.24      6231
severe_delay       0.04      0.01      0.01       199
   cancelled       0.00      0.00      0.00       510

    accuracy                           0.71     52862
   macro avg       0.28      0.29      0.27     52862
weighted avg       0.79      0.71      0.74     52862



## 6. Compare validation models and evaluate the winner once on 2025


In [20]:
results = []
for name,pred in [('majority',majority_pred),('logistic_regression',val_pred_lr),('random_forest',val_pred_rf)]:
    results.append({'model':name,'accuracy':accuracy_score(y_val,pred),'balanced_accuracy':balanced_accuracy_score(y_val,pred),'macro_f1':f1_score(y_val,pred,average='macro'),'weighted_f1':f1_score(y_val,pred,average='weighted')})
results_df = pd.DataFrame(results).sort_values('macro_f1',ascending=False)
display(results_df.round(4))

lr_f1 = f1_score(y_val,val_pred_lr,average='macro')
rf_f1 = f1_score(y_val,val_pred_rf,average='macro')
best_name,best_model = ('logistic_regression',logreg) if lr_f1 >= rf_f1 else ('random_forest',rf)
print('Selected model:',best_name)
test_pred = best_model.predict(X_test)
test_prob = best_model.predict_proba(X_test)
print('Test accuracy:',round(accuracy_score(y_test,test_pred),4))
print('Test balanced accuracy:',round(balanced_accuracy_score(y_test,test_pred),4))
print('Test macro-F1:',round(f1_score(y_test,test_pred,average='macro'),4))
print('Test log loss:',round(log_loss(y_test,test_prob,labels=best_model.classes_),4))
print(classification_report(y_test,test_pred,labels=target_classes,zero_division=0))


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
2,random_forest,0.7106,0.2872,0.2685,0.7440
0,majority,0.8687,0.2500,0.2324,0.8077
1,logistic_regression,0.3371,0.3720,0.1834,0.4326


Selected model: random_forest
Test accuracy: 0.6567
Test balanced accuracy: 0.2856
Test macro-F1: 0.2556
Test log loss: 0.749
              precision    recall  f1-score   support

      normal       0.89      0.70      0.78     29188
       delay       0.17      0.45      0.24      3996
severe_delay       0.00      0.00      0.00       169
   cancelled       0.00      0.00      0.00       270

    accuracy                           0.66     33623
   macro avg       0.26      0.29      0.26     33623
weighted avg       0.79      0.66      0.71     33623



## Interpretation

These are baselines, not the final FlightRescue AI model. The next stage should compare a stronger gradient-boosted tree model, tune only on 2024, assess probability calibration, and preserve 2025 as the untouched final test period.
